<h1 style="color: #1f77b4;">Notebook 01 — Data Collection & Benchmarks</h1>

Loads raw price data for 11 US assets (2005–2026), adds volatility signals
(VIX, MOVE), computes returns and benchmarks, and analyses performance
across five historical crisis periods.

**Parts:**
- <span style="color: #d35400; font-weight: bold;">A — Universe:</span> load, clean, and validate raw price data
- <span style="color: #d35400; font-weight: bold;">B — Signals:</span> load and align VIX and MOVE indexes
- <span style="color: #d35400; font-weight: bold;">C — Returns:</span> log returns and summary statistics
- <span style="color: #d35400; font-weight: bold;">D — Benchmarks:</span> Equal-Weight passive vs SPY buy-and-hold
- <span style="color: #d35400; font-weight: bold;">E — Crisis subsets:</span> slice data across five stress periods
- <span style="color: #d35400; font-weight: bold;">F — Results:</span> cumulative wealth, drawdowns, and comparative analysis

<h2 style="color: #f39c12;">Part A — Universe: Price Data Collection & Quality Control</h2>

Loads adjusted close prices for 11 assets from CSV files, aligns them to
a common date range (2005-02-25 → 2026-04-17), and validates data integrity.

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# Raw data path
DATA_PATH = "data/raw"

HELD_ASSETS = ['SPY', 'QQQ', 'JPM', 'XLI', 'JNJ', 'KO', 'NEE', 'IEF', 'GLD', 'XLE', 'UPS']

file_map = {
    'SPY': 'spy_us_d.csv',
    'QQQ': 'qqq_us_d.csv',
    'JPM': 'jpm_us_d.csv',
    'XLI': 'xli_us_d.csv',
    'JNJ': 'jnj_us_d.csv',
    'KO':  'ko_us_d.csv',
    'NEE': 'nee_us_d.csv',
    'IEF': 'ief_us_d.csv',
    'GLD': 'gld_us_d.csv',
    'XLE': 'xle_us_d.csv',
    'UPS': 'ups_us_d.csv',
}

In [ ]:
# Load and clean each price series first
frames = {}

for ticker, fname in file_map.items():
    df = pd.read_csv(
        os.path.join(DATA_PATH, fname),
        parse_dates=['Date']
    )

    # Keep only needed columns
    df = df[['Date', 'Close']].copy()

    # Ensure numeric close
    df['Close'] = pd.to_numeric(df['Close'], errors='coerce')

    # Sort and remove duplicate dates
    df = df.sort_values('Date')
    df = df.drop_duplicates(subset='Date', keep='last')

    # Remove invalid prices
    df = df[df['Close'] > 0]

    # Store as named series
    series = df.set_index('Date')['Close']
    series.name = ticker
    frames[ticker] = series

prices_raw = pd.DataFrame(frames)
prices_raw.index.name = 'Date'

print("Raw merged shape:", prices_raw.shape)
print("Raw date range:", prices_raw.index.min().date(), "->", prices_raw.index.max().date())
prices_raw.head(100)

<h3 style="color: #27ae60;"> Part A <br> 1. Quality Checks </h3>

Examines the raw price panel for structural issues before finalising the dataset:
- **Missing values:** counts NaNs per asset and identifies affected dates
- **Coverage:** verifies the complete-row ratio across the full date range
- **Date gaps:** flags calendar gaps larger than 4 days (expected: weekends = 1–3 days)

In [ ]:
# Diagnostic view before final cleaning
subset = prices_raw.loc['2005-02-25':'2026-04-17'].copy()

print("Subset shape before dropna:", subset.shape)
print("Subset date range:", subset.index.min().date(), "->", subset.index.max().date())

print("\nMissing values per asset:")
print(subset.isna().sum().sort_values())

complete_rows = subset.dropna(how='any')
print("\nComplete rows:", len(complete_rows))
print("Complete-date ratio:", round(len(complete_rows) / len(subset), 4))

missing_rows = subset[subset.isna().any(axis=1)]
print("\nRows with at least one missing value:", len(missing_rows))

missing_rows.head(10)

In [ ]:
# Final clean common price panel
prices = (
    prices_raw
    .loc['2005-02-25':'2026-04-17']
    .sort_index()
    .dropna(how='any')
)

print("Clean prices shape:", prices.shape)
print("Clean date range:", prices.index.min().date(), "->", prices.index.max().date())
print("Total missing values:", prices.isna().sum().sum())

prices.head()

In [ ]:
# Date gap check
date_gaps = prices.index.to_series().diff().value_counts().sort_index()

print(date_gaps.head(10))

In [ ]:
# Final summary
summary = pd.DataFrame({
    'missing_before_cleaning': subset.isna().sum(),
    'non_null_after_cleaning': prices.notna().sum(),
})

summary.loc['TOTAL'] = [
    summary['missing_before_cleaning'].sum(),
    summary['non_null_after_cleaning'].sum()
]

summary

In [ ]:
missing_rows = subset[subset.isna().any(axis=1)]
missing_rows

<h3 style="color: #27ae60;"> Part A <br> 2. Anomaly Detection </h3>

Two targeted checks to catch data errors that `dropna()` would silently miss:
- **Extreme daily returns (>30%):** flags potential data errors or unadjusted corporate actions
- **Split-like price jumps (<0.6× or >1.8× previous close):** catches unadjusted stock splits

In [ ]:
big_gaps = prices.index.to_series().diff()
big_gaps = big_gaps[big_gaps > pd.Timedelta('4 days')]
print(big_gaps)

In [ ]:
# Extreme return inspection
returns = prices.pct_change().dropna()

extreme_mask = returns.abs() > 0.30
extreme_counts = extreme_mask.sum()

print("Extreme daily moves (> 30%) per asset:")
print(extreme_counts)

extreme_rows = returns[extreme_mask.any(axis=1)]
extreme_rows

In [ ]:
# Split-like jump inspection
ratio = prices / prices.shift(1)

split_mask = (ratio < 0.6) | (ratio > 1.8)
split_rows = ratio[split_mask.any(axis=1)]

print("Potential split or abnormal jump rows:")
print(split_rows)

split_rows

<h3 style="color: #27ae60;"> Part A <br> 3. Data Quality Summary </h3>

- **Data Integrity:** Zero extreme daily returns (>30%) and zero unadjusted split jumps.
- **Missing Values:** Only 6 NaNs on 2011-02-17 (Presidents' Day). Handled with `dropna()`.
- **Trading Gaps:** Two >4-day gaps detected — both matching documented NYSE closures
  (New Year's 2007-01-03, Hurricane Sandy 2012-10-31).

In [ ]:
os.makedirs('outputs/01_data_collection', exist_ok=True)

ax = prices.plot(figsize=(14, 5), title='Price series - all assets', logy=True)
ax.set_ylabel("Price (log scale)")
plt.tight_layout()
plt.savefig('outputs/01_data_collection/01_universe_price_panel_quality_check.png', dpi=150)
plt.show()

In [ ]:
# Final quality summary
quality_summary = pd.DataFrame({
    "missing_values": subset.isna().sum(),
    "extreme_return_days": extreme_mask.sum(),
    "split_like_days": split_mask.sum()
}).sort_index()

quality_summary.loc["TOTAL"] = [
    quality_summary["missing_values"].sum(),
    quality_summary["extreme_return_days"].sum(),
    quality_summary["split_like_days"].sum()
]

quality_summary

In [ ]:
os.makedirs('data/processed', exist_ok=True)

prices.to_csv("data/processed/prices_clean.csv")
print("Saved prices_clean.csv:", prices.shape)

<h2 style="color: #f39c12;">Part B — Signals: VIX & MOVE Volatility Indexes</h2>

Loads and aligns two market volatility signals to the price panel index:
- **VIX** (CBOE Volatility Index): measures implied volatility of S&P 500 options — equity fear gauge
- **MOVE** (ICE BofA MOVE Index): measures implied volatility of US Treasury options — bond fear gauge

Both are forward-filled to match trading days and saved as `signals.csv`.

In [ ]:
VIX_PATH  = "data/raw/VIX_History.csv"
MOVE_PATH = "data/raw/MOVE_History.csv"

#  VIX 
vix = pd.read_csv(VIX_PATH)
vix.columns = [c.strip().upper() for c in vix.columns]

vix = vix.rename(columns={
    "DATE": "Date",
    "CLOSE": "VIX"
})

vix["Date"] = pd.to_datetime(vix["Date"], errors="coerce")
vix["VIX"]  = pd.to_numeric(vix["VIX"], errors="coerce")

vix = (
    vix[["Date", "VIX"]]
    .dropna()
    .drop_duplicates(subset="Date")
    .sort_values("Date")
    .set_index("Date")
)

#  MOVE 
move = pd.read_csv(MOVE_PATH)
move.columns = [c.strip() for c in move.columns]

move = move.rename(columns={
    "Date": "Date",
    "Price": "MOVE"
})

move["Date"] = pd.to_datetime(move["Date"], errors="coerce")
move["MOVE"] = pd.to_numeric(move["MOVE"], errors="coerce")

move = (
    move[["Date", "MOVE"]]
    .dropna()
    .drop_duplicates(subset="Date")
    .sort_values("Date")
    .set_index("Date")
)

print("VIX loaded :", vix.shape,  "|", vix.index.min().date(),  "->", vix.index.max().date())
print("MOVE loaded:", move.shape, "|", move.index.min().date(), "->", move.index.max().date())

<h3 style="color: #27ae60;">Part B <br> 1. Signal Quality Check</h3>

Concatenates VIX and MOVE into a single raw signals panel and inspects
coverage: first/last valid date, missing count, and missing percentage per signal.
The high missing % for MOVE before alignment is expected — its history starts in 2005
while VIX goes back to 1990.

In [ ]:
signals_raw = pd.concat([vix, move], axis=1).sort_index()

print("signals_raw shape:", signals_raw.shape)
print("signals_raw date range:", signals_raw.index.min().date(), "->", signals_raw.index.max().date())
print("\nMissing values per column:")
print(signals_raw.isna().sum())

signals_qc = pd.DataFrame({
    "first_date": signals_raw.apply(lambda s: s.first_valid_index()),
    "last_date": signals_raw.apply(lambda s: s.last_valid_index()),
    "missing_count": signals_raw.isna().sum(),
    "missing_pct": (signals_raw.isna().mean() * 100).round(2)
})

display(signals_qc)

<h3 style="color: #27ae60;">Part B <br> 2. Alignment & Forward-Fill</h3>

Reindexes signals to the price panel's trading-day calendar and forward-fills
any remaining gaps (e.g. holidays where VIX/MOVE have no observation).
Rows with leading NaNs (before first valid signal date) are dropped.

In [ ]:
signals = signals_raw.reindex(prices.index).ffill()

# Remove any leading rows that still have NaN after alignment
signals = signals.dropna(how="any")

# Keep only common dates between prices and signals
prices_aligned = prices.loc[signals.index].copy()

print("prices_aligned shape :", prices_aligned.shape)
print("signals shape        :", signals.shape)
print("Common date range    :", signals.index.min().date(), "->", signals.index.max().date())
print("Remaining signal NaN :", signals.isna().sum().sum())

<h3 style="color: #27ae60;">Part B <br> 3. Master Panel</h3>

Merges aligned prices and signals into a single master panel.
Three assertions verify integrity before export:
monotonic index, zero NaNs in signals, and presence of VIX and MOVE columns.

In [ ]:
master_panel = pd.concat([prices_aligned, signals], axis=1)

assert master_panel.index.is_monotonic_increasing
assert signals.isna().sum().sum() == 0
assert set(["VIX", "MOVE"]).issubset(master_panel.columns)

signals.to_csv("data/processed/signals.csv")
master_panel.to_csv("data/processed/master_prices_with_signals.csv")

print("master_panel shape:", master_panel.shape)
print("Saved: data/processed/signals.csv")
print("Saved: data/processed/master_prices_with_signals.csv")

display(master_panel.head())

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 7), sharex=True)

signals["VIX"].plot(ax=axes[0], title="VIX — Aligned Signal", color="tab:red")
axes[0].set_ylabel("Level")
axes[0].grid(True, alpha=0.3)

signals["MOVE"].plot(ax=axes[1], title="MOVE — Aligned Signal", color="tab:blue")
axes[1].set_ylabel("Level")
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("outputs/01_data_collection/02_signals_overview.png", dpi=150)
plt.show()

print("Saved: outputs/01_data_collection/02_signals_overview.png")

<h2 style="color: #f39c12;">Part C — Returns: Log Returns & Summary Statistics</h2>

Converts the clean price panel into daily log returns and computes
full-period annualised statistics per asset.
Log returns are preferred over simple returns for time-series analysis
because they are additive across time, approximately normally distributed,
and numerically stable for compounding over long horizons.

<h3 style="color: #27ae60;">Part C <br> 1. Daily Log Returns</h3>

Computes daily log returns as:

$$r_t = \ln\left(\frac{P_t}{P_{t-1}}\right)$$

A single `assert` confirms zero NaNs — any surviving NaN would silently
corrupt all downstream statistics and portfolio calculations.

In [ ]:
# Log returns: ln(P_t / P_{t-1})
log_returns = np.log(prices / prices.shift(1)).dropna()

assert log_returns.isna().sum().sum() == 0, "NaNs found in returns!"
print("Returns shape:", log_returns.shape)
print("Date range:", log_returns.index.min().date(), "→", log_returns.index.max().date())

log_returns.head()

<h3 style="color: #27ae60;">Part C <br> 2. Full-Period Summary Statistics</h3>

Computes annualised statistics for each asset over the entire sample (2005–2026).
Key metrics and why they matter:

- **Annualised mean & vol:** scaled by √252 to make daily figures comparable across assets
- **Sharpe ratio:** risk-adjusted return — higher is better; allows comparison across assets with different volatility profiles
- **Skewness:** negative skew means fat left tail (crash risk); most equity assets are negatively skewed
- **Excess kurtosis:** values >> 0 indicate fat tails — returns are not normally distributed, which matters for risk models
- **Max drawdown:** worst peak-to-trough loss over the full period — a direct measure of downside risk experienced by a buy-and-hold investor

In [ ]:
TRADING_DAYS = 252

def summarise(ret: pd.DataFrame, price_df: pd.DataFrame, ann: int = TRADING_DAYS) -> pd.DataFrame:
    """Annualised stats from daily log returns."""
    stats = pd.DataFrame(index=ret.columns)
    stats["ann_mean_%"] = (ret.mean() * ann * 100).round(2)
    stats["ann_vol_%"] = (ret.std() * np.sqrt(ann) * 100).round(2)
    stats["sharpe"] = (ret.mean() / ret.std() * np.sqrt(ann)).round(3)
    stats["skewness"] = ret.skew().round(3)
    stats["kurtosis"] = ret.kurt().round(3)  # excess kurtosis
    stats["max_drawdown_%"] = (((price_df / price_df.cummax()) - 1).min() * 100).round(2)
    return stats

stats_full = summarise(log_returns, prices)
print(stats_full.to_string())

In [ ]:
os.makedirs("data/processed", exist_ok=True)
os.makedirs("outputs/01_data_collection", exist_ok=True)

log_returns.to_csv("data/processed/log_returns.csv")
stats_full.to_csv("outputs/01_data_collection/03_1_returns_summary_stats.csv")

print("Saved log_returns.csv", log_returns.shape)
print("Saved returns_summary_stats.csv")

<h3 style="color: #27ae60;">Part C <br> 3. Cumulative Wealth — Sanity Check</h3>

Reconstructs cumulative wealth from log returns as:

$$W_t = \exp\left(\sum_{s=1}^{t} r_s\right)$$

Plotted on a **log scale** so that equal vertical distances represent equal
percentage changes — this makes long-horizon comparisons visually fair
regardless of absolute price levels. Used here purely as a visual sanity
check before proceeding to formal benchmarks in Part D.

In [ ]:
cum = np.exp(log_returns.cumsum())

fig, ax = plt.subplots(figsize=(14, 5))
cum.plot(ax=ax, logy=True, alpha=0.8)
ax.set_title("Cumulative Wealth — all assets (log scale)")
ax.set_ylabel("Growth of $1 (log)")
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("outputs/01_data_collection/03_2_cumulative_wealth_sanity.png", dpi=150)
plt.show()
print("Saved cumulative wealth plot")

part D

In [ ]:
ROLLING_WINDOW = 252

assert "prices" in globals(), "prices dataframe not found."
assert "HELD_ASSETS" in globals(), "HELD_ASSETS not found."

benchmark_prices = prices[HELD_ASSETS].copy()
simple_returns = benchmark_prices.pct_change(fill_method=None)

print("benchmark_prices shape:", benchmark_prices.shape)
print("simple_returns shape  :", simple_returns.shape)
print("date range            :", benchmark_prices.index.min().date(), "->", benchmark_prices.index.max().date())
print("NaNs in simple_returns:", int(simple_returns.isna().sum().sum()))

In [ ]:
def build_equal_weight_passive(price_df: pd.DataFrame) -> pd.DataFrame:
    """Build passive equal-weight portfolio from price panel."""
    normalized = price_df.div(price_df.iloc[0])      # base = 1 at inception
    wealth = normalized.mean(axis=1)                 # equal initial split
    daily_return = wealth.pct_change(fill_method=None)

    out = pd.DataFrame({
        "ew_wealth": wealth,
        "ew_return": daily_return
    })

    return out

ew_portfolio = build_equal_weight_passive(benchmark_prices)

print("EW portfolio shape:", ew_portfolio.shape)
print("EW date range     :", ew_portfolio.index.min().date(), "->", ew_portfolio.index.max().date())
display(ew_portfolio.head())

In [ ]:
spy_benchmark = pd.DataFrame(index=benchmark_prices.index)
spy_benchmark["spy_wealth"] = benchmark_prices["SPY"] / benchmark_prices["SPY"].iloc[0]
spy_benchmark["spy_return"] = spy_benchmark["spy_wealth"].pct_change(fill_method=None)

print("SPY benchmark shape:", spy_benchmark.shape)
display(spy_benchmark.head())

In [ ]:
benchmarks = pd.concat(
    [
        ew_portfolio[["ew_wealth", "ew_return"]],
        spy_benchmark[["spy_wealth", "spy_return"]]
    ],
    axis=1
).sort_index()

print("benchmarks shape:", benchmarks.shape)
print("date range      :", benchmarks.index.min().date(), "->", benchmarks.index.max().date())
print("\nNaN count per column:")
print(benchmarks.isna().sum())
print("\nNote: the first row has NaN returns due to pct_change() base — this is expected.")

display(benchmarks.head())

In [ ]:

def rolling_sharpe(ret: pd.Series, window: int = ROLLING_WINDOW, ann: int = TRADING_DAYS) -> pd.Series:
    """Rolling annualised Sharpe ratio."""
    rolling_mean = ret.rolling(window).mean()
    rolling_std = ret.rolling(window).std()
    sharpe = (rolling_mean / rolling_std) * np.sqrt(ann)
    return sharpe

def drawdown_from_wealth(wealth: pd.Series) -> pd.Series:
    """Drawdown from wealth index."""
    running_max = wealth.cummax()
    dd = wealth / running_max - 1.0
    return dd

benchmarks["ew_rolling_sharpe"] = rolling_sharpe(benchmarks["ew_return"])
benchmarks["spy_rolling_sharpe"] = rolling_sharpe(benchmarks["spy_return"])

benchmarks["ew_drawdown"] = drawdown_from_wealth(benchmarks["ew_wealth"])
benchmarks["spy_drawdown"] = drawdown_from_wealth(benchmarks["spy_wealth"])

benchmarks[[
    "ew_rolling_sharpe", "spy_rolling_sharpe",
    "ew_drawdown", "spy_drawdown"
]].tail()

In [ ]:
def summarise_benchmark(ret: pd.Series, wealth: pd.Series, ann: int = TRADING_DAYS) -> dict:
    """Summary stats for a benchmark series."""
    ret_clean = ret.dropna()
    dd = drawdown_from_wealth(wealth)

    return {
        "ann_mean_%": round(ret_clean.mean() * ann * 100, 2),
        "ann_vol_%": round(ret_clean.std() * np.sqrt(ann) * 100, 2),
        "sharpe": round((ret_clean.mean() / ret_clean.std()) * np.sqrt(ann), 3),
        "final_wealth": round(wealth.iloc[-1], 4),
        "total_return_%": round((wealth.iloc[-1] - 1) * 100, 2),
        "max_drawdown_%": round(dd.min() * 100, 2),
    }

benchmark_summary = pd.DataFrame({
    "Equal_Weight_Passive": summarise_benchmark(
        benchmarks["ew_return"], benchmarks["ew_wealth"]
    ),
    "SPY_Buy_and_Hold": summarise_benchmark(
        benchmarks["spy_return"], benchmarks["spy_wealth"]
    )
}).T

display(benchmark_summary)

In [ ]:
os.makedirs("data/processed", exist_ok=True)
os.makedirs("outputs/01_data_collection", exist_ok=True)

benchmarks.to_csv("data/processed/benchmarks.csv")
benchmark_summary.to_csv("outputs/01_data_collection/04_benchmark_summary.csv")

print("Saved: data/processed/benchmarks.csv")
print("Saved: outputs/01_data_collection/04_benchmark_summary.csv")

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))

benchmarks[["ew_wealth", "spy_wealth"]].plot(ax=ax, linewidth=2)
ax.set_title("Benchmark Wealth Comparison")
ax.set_ylabel("Growth of $1")
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("outputs/01_data_collection/05_benchmark_wealth.png", dpi=150)
plt.show()

print("Saved: outputs/01_data_collection/05_benchmark_wealth.png")

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))

benchmarks[["ew_rolling_sharpe", "spy_rolling_sharpe"]].plot(ax=ax, linewidth=1.8)
ax.set_title("Rolling 252-Day Sharpe Ratio")
ax.set_ylabel("Sharpe")
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("outputs/01_data_collection/06_benchmark_rolling_sharpe.png", dpi=150)
plt.show()

print("Saved: outputs/01_data_collection/06_benchmark_rolling_sharpe.png")

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))

benchmarks["ew_drawdown"].plot(ax=ax, linewidth=1.8, label="EW drawdown")
benchmarks["spy_drawdown"].plot(ax=ax, linewidth=1.8, label="SPY drawdown")

ax.fill_between(
    benchmarks.index,
    benchmarks["ew_drawdown"],
    0,
    alpha=0.18,
    label="_nolegend_"
)

ax.fill_between(
    benchmarks.index,
    benchmarks["spy_drawdown"],
    0,
    alpha=0.18,
    label="_nolegend_"
)

ax.set_title("Benchmark Drawdown Series")
ax.set_ylabel("Drawdown")
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("outputs/01_data_collection/07_benchmark_drawdowns.png", dpi=150)
plt.show()

print("Saved: outputs/01_data_collection/07_benchmark_drawdowns.png")

Part E

In [ ]:
CRISIS_WINDOWS = {
    "GFC_2008":           ("2007-10-01", "2009-03-31"),
    "EU_Debt_2011":       ("2010-04-01", "2012-06-30"),
    "COVID_2020":         ("2020-02-01", "2020-06-30"),
    "Rate_Hike_2022":     ("2022-01-01", "2022-12-31"),
    "Trump_Tariffs_2025": ("2025-01-01", "2026-04-17"),
}

for name, (start, end) in CRISIS_WINDOWS.items():
    n = len(benchmarks.loc[start:end])
    print(f"{name:25s}  {start} → {end}  ({n} trading days)")

In [ ]:
def slice_crisis(df: pd.DataFrame, start: str, end: str) -> pd.DataFrame:
    """Slice DataFrame to crisis window (inclusive)."""
    return df.loc[start:end].copy()

crisis_prices  = {}
crisis_signals = {}
crisis_returns = {}
crisis_bench   = {}

for name, (start, end) in CRISIS_WINDOWS.items():
    crisis_prices[name]  = slice_crisis(benchmark_prices, start, end)
    crisis_signals[name] = slice_crisis(signals, start, end)
    crisis_returns[name] = slice_crisis(simple_returns, start, end)
    crisis_bench[name]   = slice_crisis(benchmarks, start, end)

print("Crisis slices:")
for name, df in crisis_returns.items():
    print(f"  {name:25s}  {df.shape[0]:4d} days  "
          f"{df.index.min().date()} → {df.index.max().date()}")

In [ ]:
def summarise_crisis(ret_df: pd.DataFrame,
                     price_df: pd.DataFrame,
                     ann: int = TRADING_DAYS) -> pd.DataFrame:
    """Annualised stats + max drawdown for each asset in a crisis window."""
    ret   = ret_df.dropna()
    stats = pd.DataFrame(index=ret.columns)
    stats["total_%"]    = ((ret + 1).prod() - 1).mul(100).round(2)
    stats["ann_mean_%"] = (ret.mean() * ann * 100).round(2)
    stats["ann_vol_%"]  = (ret.std() * np.sqrt(ann) * 100).round(2)
    stats["sharpe"]     = (ret.mean() / ret.std() * np.sqrt(ann)).round(3)
    stats["max_dd_%"]   = (
        ((price_df / price_df.cummax()) - 1).min() * 100
    ).round(2)
    return stats

crisis_stats = {
    name: summarise_crisis(crisis_returns[name], crisis_prices[name])
    for name in CRISIS_WINDOWS
}

print("=== GFC 2008 ===")
display(crisis_stats["GFC_2008"])

In [ ]:
def summarise_bench_crisis(bench_df: pd.DataFrame,
                           ann: int = TRADING_DAYS) -> pd.DataFrame:
    """EW vs SPY summary for a single crisis window."""
    rows = {}
    for label, ret_col, wealth_col in [
        ("EW_Passive",  "ew_return",  "ew_wealth"),
        ("SPY_BuyHold", "spy_return", "spy_wealth"),
    ]:
        ret    = bench_df[ret_col].dropna()
        wealth = bench_df[wealth_col]
        dd     = drawdown_from_wealth(wealth / wealth.iloc[0])  # re-base to 1
        rows[label] = {
            "total_%":        round((wealth.iloc[-1] / wealth.iloc[0] - 1) * 100, 2),
            "ann_vol_%":      round(ret.std() * np.sqrt(ann) * 100, 2),
            "sharpe":         round(ret.mean() / ret.std() * np.sqrt(ann), 3),
            "max_drawdown_%": round(dd.min() * 100, 2),
        }
    return pd.DataFrame(rows).T

crisis_bench_summary = {
    name: summarise_bench_crisis(df)
    for name, df in crisis_bench.items()
}

print("=== COVID 2020 ===")
display(crisis_bench_summary["COVID_2020"])

In [ ]:
os.makedirs("outputs/01_data_collection", exist_ok=True)
os.makedirs("data/processed/crisis", exist_ok=True)

for name in CRISIS_WINDOWS:
    crisis_prices[name].to_csv(
        f"data/processed/crisis/08_1_{name}_prices.csv")
    crisis_signals[name].to_csv(
        f"data/processed/crisis/08_2_{name}_signals.csv")
    crisis_returns[name].to_csv(
        f"data/processed/crisis/08_3_{name}_returns.csv")
    crisis_stats[name].to_csv(
        f"data/processed/crisis/08_4_{name}_asset_stats.csv")
    crisis_bench_summary[name].to_csv(
        f"outputs/01_data_collection/08_5_{name}_bench_summary.csv")

print("All crisis files saved:")
for name in CRISIS_WINDOWS:
    print(f"  ✓ {name}")

In [ ]:
total_ret_matrix = pd.DataFrame({
    name: crisis_stats[name]["total_%"]
    for name in CRISIS_WINDOWS
})

fig, ax = plt.subplots(figsize=(14, 6))
total_ret_matrix.T.plot(kind="bar", ax=ax, width=0.75)
ax.axhline(0, color="black", linewidth=0.8)
ax.set_title("Total Return per Asset — Crisis Periods")
ax.set_ylabel("Total Return (%)")
ax.legend(bbox_to_anchor=(1.01, 1), loc="upper left", fontsize=8)
ax.tick_params(axis="x", rotation=20)
ax.grid(True, axis="y", alpha=0.3)

plt.tight_layout()
plt.savefig("outputs/01_data_collection/09_crisis_asset_returns.png", dpi=150)
plt.show()
print("Saved: 09_crisis_asset_returns.png")

In [ ]:
dd_matrix = pd.DataFrame({
    name: crisis_stats[name]["max_dd_%"]
    for name in CRISIS_WINDOWS
})

fig, ax = plt.subplots(figsize=(14, 6))
dd_matrix.T.plot(kind="bar", ax=ax, width=0.75)
ax.axhline(0, color="black", linewidth=0.8)
ax.set_title("Max Drawdown per Asset — Crisis Periods")
ax.set_ylabel("Max Drawdown (%)")
ax.legend(bbox_to_anchor=(1.01, 1), loc="upper left", fontsize=8)
ax.tick_params(axis="x", rotation=20)
ax.grid(True, axis="y", alpha=0.3)

plt.tight_layout()
plt.savefig("outputs/01_data_collection/10_crisis_asset_drawdowns.png", dpi=150)
plt.show()
print("Saved: 10_crisis_asset_drawdowns.png")

In [ ]:
bench_total = pd.DataFrame({
    name: crisis_bench_summary[name]["total_%"]
    for name in CRISIS_WINDOWS
}).T

fig, ax = plt.subplots(figsize=(10, 5))
bench_total.plot(kind="bar", ax=ax, width=0.6,
                 color=["steelblue", "orange"])
ax.axhline(0, color="black", linewidth=0.8)
ax.set_title("EW vs SPY — Total Return per Crisis Period")
ax.set_ylabel("Total Return (%)")
ax.tick_params(axis="x", rotation=20)
ax.grid(True, axis="y", alpha=0.3)

plt.tight_layout()
plt.savefig("outputs/01_data_collection/11_crisis_bench_comparison.png", dpi=150)
plt.show()
print("Saved: 11_crisis_bench_comparison.png")

Part F 

In [ ]:
n_crises = len(CRISIS_WINDOWS)
fig, axes = plt.subplots(1, n_crises, figsize=(18, 4), sharey=False)

for ax, (name, (start, end)) in zip(axes, CRISIS_WINDOWS.items()):
    bench = crisis_bench[name]

    # Re-base both to 1 at crisis start
    ew_rebased  = bench["ew_wealth"]  / bench["ew_wealth"].iloc[0]
    spy_rebased = bench["spy_wealth"] / bench["spy_wealth"].iloc[0]

    ew_rebased.plot(ax=ax, label="EW",  color="steelblue", linewidth=1.8)
    spy_rebased.plot(ax=ax, label="SPY", color="orange",    linewidth=1.8)

    ax.axhline(1.0, color="black", linewidth=0.7, linestyle="--")
    ax.set_title(name.replace("_", "\n"), fontsize=9)
    ax.set_ylabel("Wealth (rebased=1)" if ax == axes[0] else "")
    ax.grid(True, alpha=0.3)
    ax.tick_params(axis="x", rotation=30, labelsize=7)

axes[-1].legend(loc="best", fontsize=8)
fig.suptitle("EW vs SPY — Cumulative Wealth per Crisis (rebased to 1)", y=1.02)

plt.tight_layout()
plt.savefig("outputs/01_data_collection/12_crisis_wealth_subplots.png",
            dpi=150, bbox_inches="tight")
plt.show()
print("Saved: 12_crisis_wealth_subplots.png")

In [ ]:
fig, axes = plt.subplots(1, n_crises, figsize=(18, 4), sharey=True)

for ax, (name, (start, end)) in zip(axes, CRISIS_WINDOWS.items()):
    bench = crisis_bench[name]

    ew_dd  = drawdown_from_wealth(bench["ew_wealth"]  / bench["ew_wealth"].iloc[0])
    spy_dd = drawdown_from_wealth(bench["spy_wealth"] / bench["spy_wealth"].iloc[0])

    ew_dd.plot(ax=ax,  label="EW",  color="steelblue", linewidth=1.5)
    spy_dd.plot(ax=ax, label="SPY", color="orange",    linewidth=1.5)
    ax.fill_between(bench.index, ew_dd,  0, alpha=0.15, color="steelblue")
    ax.fill_between(bench.index, spy_dd, 0, alpha=0.15, color="orange")

    ax.axhline(0, color="black", linewidth=0.7)
    ax.set_title(name.replace("_", "\n"), fontsize=9)
    ax.set_ylabel("Drawdown" if ax == axes[0] else "")
    ax.grid(True, alpha=0.3)
    ax.tick_params(axis="x", rotation=30, labelsize=7)

axes[-1].legend(loc="best", fontsize=8)
fig.suptitle("EW vs SPY — Drawdown per Crisis", y=1.02)

plt.tight_layout()
plt.savefig("outputs/01_data_collection/13_crisis_drawdown_subplots.png",
            dpi=150, bbox_inches="tight")
plt.show()
print("Saved: 13_crisis_drawdown_subplots.png")

In [ ]:
# Build one table: crisis × [EW total%, EW maxdd%, SPY total%, SPY maxdd%, EW Sharpe, SPY Sharpe]
rows = []
for name in CRISIS_WINDOWS:
    s = crisis_bench_summary[name]
    rows.append({
        "Crisis":           name,
        "Days":             len(crisis_bench[name]),
        "EW_total_%":       s.loc["EW_Passive",  "total_%"],
        "EW_maxdd_%":       s.loc["EW_Passive",  "max_drawdown_%"],
        "EW_sharpe":        s.loc["EW_Passive",  "sharpe"],
        "SPY_total_%":      s.loc["SPY_BuyHold", "total_%"],
        "SPY_maxdd_%":      s.loc["SPY_BuyHold", "max_drawdown_%"],
        "SPY_sharpe":       s.loc["SPY_BuyHold", "sharpe"],
        "EW_vs_SPY_%":      round(
            s.loc["EW_Passive", "total_%"] - s.loc["SPY_BuyHold", "total_%"], 2
        ),
    })

report_table = pd.DataFrame(rows).set_index("Crisis")
display(report_table)

In [ ]:
report_table.to_csv("outputs/01_data_collection/14_final_crisis_report.csv")
print("Saved: outputs/01_data_collection/14_final_crisis_report.csv")

# Full-period benchmark summary reminder
print("\n=== Full-period benchmark summary ===")
display(benchmark_summary)

In [ ]:
total_ret_matrix = pd.DataFrame({
    name: crisis_stats[name]["total_%"]
    for name in CRISIS_WINDOWS
})  # shape: assets × crises

vmax = np.ceil(np.abs(total_ret_matrix.values).max() / 10) * 10
vmin = -vmax

fig, ax = plt.subplots(figsize=(12, 6))
im = ax.imshow(
    total_ret_matrix.values,
    cmap="RdYlGn",
    aspect="auto",
    vmin=vmin,
    vmax=vmax
)

ax.set_xticks(range(len(CRISIS_WINDOWS)))
ax.set_xticklabels([n.replace("_", "\n") for n in CRISIS_WINDOWS], fontsize=9)
ax.set_yticks(range(len(total_ret_matrix.index)))
ax.set_yticklabels(total_ret_matrix.index, fontsize=9)

for i in range(len(total_ret_matrix.index)):
    for j in range(len(CRISIS_WINDOWS)):
        val = total_ret_matrix.values[i, j]
        ax.text(
            j, i, f"{val:.1f}%",
            ha="center", va="center",
            fontsize=8, color="black"
        )

plt.colorbar(im, ax=ax, label="Total Return (%)")
ax.set_title("Asset Total Return Heatmap — Crisis Periods")

plt.tight_layout()
plt.savefig("outputs/01_data_collection/15_crisis_heatmap.png", dpi=150)
plt.show()
print("Saved: 15_crisis_heatmap.png")